## setup

In [ ]:
# import stuff
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# load data from folder
folder = Path('../data')

# filtered dataset with mortgage rates
listings = pd.read_csv(folder / 'listings_with_rates.csv', low_memory = False)

# null count summary as reference for cleaning
listings_null_summary = pd.read_csv(folder / 'listings_null_summary.csv', index_col = 0)

In [ ]:
print('Listings dataset shape:', listings.shape)
listings.head()

In [ ]:
listings_null_summary.head()

## data cleaning & prep
- convert date fields to datetime format
- remove unnecessary/redundant cols
- handle missing values appropriately
- ensure numeric fields are properly typed
- remove/flag invalid numeric values

In [ ]:
# convert date columns to datetime format
date_cols = ['CloseDate',
             'PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']
listings[date_cols] = listings[date_cols].apply(pd.to_datetime, errors = 'coerce')

# check that changes have been made
listings[date_cols].dtypes

In [ ]:
# check cols w >90% nulls
flag_over_90 = listings_null_summary[listings_null_summary['null pct'] > 90].index.tolist()
flag_over_90.sort()
print(len(flag_over_90))
flag_over_90

From the ``Real_Estate_Primer.pdf``, recall that our key data fields are:
- ``ListingKey``
- ``ListingContractDate``
- ``ListPrice``
- ``ClosePrice``
- ``PurchaseContractDate``
- ``CloseDate``
- ``LivingArea``
- ``BedroomsTotal``
- ``BathroomsTotalInteger``
- ``Latitude``/``Longitude``
- ``UnparsedAddress``

Since the above columns are not part of key fields, we can remove them.

In [ ]:
# drop cols w >90% nulls
listings_clean = listings.drop(columns = flag_over_90)

print('Listings shape after dropping:', listings_clean.shape)
listings_clean.head()

### shape after dropping >90% null: (607724, 73)

We can also consider dropping columns with over 50% nulls for more meaningful analyses, but we will still make sure that none of these flagged columns are considered key data fields. From our Week 2-3 deliverable instructions, we were given a list of key numeric fields (in which I will define as ``core_fields``), so we should also make sure that they are not flagged.

In [ ]:
# consider dropping cols w >50% nulls
flag_over_50 = listings_null_summary[listings_null_summary['null pct'] > 50].index.tolist()

# recall wk2-3: remove core fields from the list of cols to drop
core_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
               'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
               'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

for field in core_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields)') # overlaps with >90% null cols
flag_over_50

In week 6, we will be feature engineering using existing columns and also adding school districts using properties' latitude and longitude values, so we will exclude removing schools and school districts in case we end up populating them in future deliverables.

In [ ]:
# remove schools and school districts from flagged cols
school_fields = ['ElementarySchool',
                 'ElementarySchoolDistrict',
                 'MiddleOrJuniorSchool',
                 'MiddleOrJuniorSchoolDistrict',
                 'HighSchool']

for field in school_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

for i in flag_over_90:
    for j in flag_over_50:
        if j in flag_over_90:
            flag_over_50.remove(j)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields & schools):')
flag_over_50

In [ ]:
# drop cols w >50% nulls (excl. core fields and schools)
listings_clean = listings_clean.drop(columns = flag_over_50)
print('Listings shape after dropping:', listings_clean.shape)

listings_clean.head()

### shape after dropping >50% null: (607724, 58)

We can take a look at the remaining columns and determine what else to remove.

In [ ]:
sorted(listings_clean.columns)

In [ ]:
# set up list to remove remaining cols
cols_to_remove = []

listings[['ListAgentFirstName', 'ListAgentFullName', 'ListAgentLastName', 'ListAgentEmail']].isna().sum()

In [ ]:
# there seems to be duplicate cols, so we should remove those too
dupe_cols = [col for col in listings_clean.columns if col.endswith('.1')]
print(dupe_cols)

for col in dupe_cols:
    cols_to_remove.append(col)

cols_to_remove

In [ ]:
# remove first & last name since there's already fullname col
cols_to_remove.append('ListAgentFirstName')
cols_to_remove.append('ListAgentLastName')

# not needed for analysis
cols_to_remove.append('ListAgentEmail')

In [ ]:
# street numbers are unique + not needed for analysis
print(listings['StreetNumberNumeric'].value_counts())

cols_to_remove.append('StreetNumberNumeric')

In [ ]:
print(listings_clean['ListingKeyNumeric'].equals(listings_clean['ListingKey']))

cols_to_remove.append('ListingKeyNumeric')

In [ ]:
print(listings_clean[['PropertyType', 'PropertySubType']].value_counts())

cols_to_remove.append('PropertyType')

In [ ]:
# print(listings_clean[['LotSizeAcres', 'LotSizeArea', 'LotSizeSquareFeet']])
print(listings_clean['LotSizeArea'].equals(listings_clean['LotSizeSquareFeet']))

cols_to_remove.append('LotSizeArea')

In [ ]:
print(listings_clean['StateOrProvince'].unique())

cols_to_remove.append('StateOrProvince')

### REMOVE COLUMNS

In [ ]:
# double check before removing
cols_to_remove

#### reviewing columns to remove

| column name | description |
| ----------- | ---------- |
| ListAgentFirstName | The first name of the listing agent |
| ListAgentLastName | The last name of the listing agent |
| StreetNumberNumeric | The integer portion of the street number |
| PropertyType | The list of types of properties such as Residential, Lease, Income, Land, Mobile, Commercial Sale, etc |
| ListAgentEmail | The email address of the Listing Agent |
| LotSizeArea | The total area of the lot. See Lot Size Units for the units of measurement (Square Feet, Square Meters, Acres, etc) |
| StateOrProvince | Text field containing the accepted postal abbreviation for the state or province | 
| ListingKeyNumeric | same as ``ListingKey`` column |

the remaining 9 columns ending with '.1' are removed since they are duplicates of existing columns

In [ ]:
# drop columns
listings_clean = listings_clean.drop(columns = cols_to_remove)

In [ ]:
print(listings_clean.shape)
print(listings_clean.columns)

### final shape after dropping: (607724, 41)

### CLEAN ROWS

now that we've dealt with columns, lets look into invalid values for ClosePrice, LivingArea, DOM, and negative bedrooms or bathrooms

In [ ]:
# flag for negative closeprice
listings_clean[listings_clean['ClosePrice'] <= 0]

In [ ]:
# flag for negative livingarea
listings_clean['neg_livingarea_flag'] = listings_clean['LivingArea'] <= 0

listings_clean[listings_clean['neg_livingarea_flag'] == True]

In [ ]:
# look @ DOM
listings_clean[['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head()

In [ ]:
# flag @ negative DOM
listings_clean['neg_dom_flag'] = listings_clean['DaysOnMarket'] < 0

print(listings_clean[listings_clean['neg_dom_flag'] == True].shape)
listings_clean[listings_clean['neg_dom_flag'] == True][['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head()

In [ ]:
# look @ negative bedrooms
print(listings_clean[listings_clean['BedroomsTotal'] < 0].shape)
listings_clean[listings_clean['BedroomsTotal'] < 0].head()

In [ ]:
# look @ negative bathrooms
listings_clean[listings_clean['BathroomsTotalInteger'] < 0]

### data consistency checks
- validate logical order of date fields (ListingContractDate should precede PurchaseContractDate which should precede CloseDate)
- create boolean flag cols to mark records that violate these rules
    - listing_after_close_flag
    - purchase_after_close_flag
    - negative_timeline_flag

In [ ]:
# validate order of datefields
listings_clean['negative_timeline_flag'] = listings_clean['ListingContractDate'] > listings_clean['PurchaseContractDate']

print('Shape where date fields are out of order', listings_clean[listings_clean['negative_timeline_flag'] == True].shape)
listings_clean[listings_clean['negative_timeline_flag'] == True][['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head()

### geographic data checks
- flag records w missing coords (lat/lon is null)
- flag lat = 0 or lon = 0 (sentinel null vals)
- flag lon > 0 errors (cal coords should be negative)
- flag out-of-state/implausible coords

In [ ]:
# flag missing coords (lat/lon is null)
listings_clean['missing_coords_flag'] = (listings_clean['Latitude'].isna()) | (listings_clean['Longitude'].isna())

print(listings_clean[listings_clean['missing_coords_flag'] == True].shape)
listings_clean[listings_clean['missing_coords_flag'] == True].head()

In [ ]:
# lat = 0 or lon = 0
listings_clean['sentinel_coords_flag'] = (listings_clean['Latitude'] == 0) | (listings_clean['Longitude'] == 0)

print(listings_clean[listings_clean['sentinel_coords_flag'] == True].shape)
listings_clean[listings_clean['sentinel_coords_flag'] == True].head()

In [ ]:
# lon > 0 errors (cali coords should be negative)
listings_clean['pos_lon_flag'] = listings_clean['Longitude'] > 0

print(listings_clean[listings_clean['pos_lon_flag'] == True].shape)
listings_clean[listings_clean['pos_lon_flag'] == True].head()

In [ ]:
# out of state (oos) or implausible coords

'''cali coords range
lat: (32.0, 42.5)
lon: (-125.0, -113.5)
'''

# filter for only cali listings
listings_clean['oos_coords_flag'] = ~(listings_clean['Latitude'].between(32.0, 42.5) & listings_clean['Longitude'].between(-125.0, -113.5))

listings_clean['oos_coords_flag'].value_counts()

In [ ]:
print('# rows with missing coordinates:', listings_clean[listings_clean['missing_coords_flag'] == True].shape[0])
print('# rows with sentinel coordinates:', listings_clean[listings_clean['sentinel_coords_flag'] == True].shape[0])

print('# rows with non-California values (negative longitude):', listings_clean[listings_clean['pos_lon_flag'] == True].shape[0])
print('# rows with out of state/implausible coordinates:', listings_clean[listings_clean['oos_coords_flag'] == True].shape[0])

In [ ]:
listings_clean.shape

### review flagged cols

We created 7 flagged columns:
- neg_livingarea_flag
- neg_dom_flag
- negative_timeline_flag
- missing_coords_flag
- sentinel_coords_flag
- pos_lon_flag
- oos_coords_flag

So now we can start cleaning by rows. Note that listing_after_close_flag and purchase_after_close_flag were not created since this dataset does not have a closedate column

In [ ]:
listings_clean.iloc[:,-8:]

In [ ]:
# look into negative livingarea values
print(listings_clean[listings_clean['neg_livingarea_flag'] == True].shape)
print(listings_clean[listings_clean['neg_livingarea_flag'] == True]['LivingArea'].head())

listings_clean[listings_clean['neg_livingarea_flag'] == True]['LivingArea'].value_counts()

In [ ]:
# remove 0-value livingarea
print('Shape before removing:', listings_clean.shape)
listings_clean = listings_clean[listings_clean['neg_livingarea_flag'] == False]

print('Shape after removing:', listings_clean.shape)
listings_clean.head(3)

In [ ]:
# set negative dom values to null
print('# rows with negative DOM:', listings_clean[listings_clean['neg_dom_flag'] == True]['DaysOnMarket'].shape)

print('# nulls before conversion:', listings_clean['DaysOnMarket'].isna().sum())
listings_clean.loc[listings_clean['neg_dom_flag'] == True, 'DaysOnMarket'] = np.nan

print('# nulls after conversion:', listings_clean['DaysOnMarket'].isna().sum())

In [ ]:
# investigate negative timeline flag
mask = listings_clean[listings_clean['negative_timeline_flag'] == True]
print(mask.shape)

mask[['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head(3)

In [ ]:
# turn wrong dates into null
print('# rows with wrong timeline:', mask[['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].shape)

print('\n# nulls before conversion:', listings_clean[['ListingContractDate', 'PurchaseContractDate']].isna().sum())
listings_clean.loc[listings_clean['negative_timeline_flag'] == True, ['ListingContractDate', 'PurchaseContractDate']] = np.nan

print('\n# nulls after conversion:', listings_clean[['ListingContractDate', 'PurchaseContractDate']].isna().sum())

In [ ]:
# investigate missing coords flag
mask = listings_clean[listings_clean['missing_coords_flag'] == True]
print('# rows with missing coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

In [ ]:
# investigate sentinel coords
mask = listings_clean[listings_clean['sentinel_coords_flag'] == True]
print('# rows with sentinel coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

In [ ]:
# remove 0 lat/lon values
print('Shape before removing:', listings_clean.shape)
listings_clean = listings_clean[listings_clean['sentinel_coords_flag'] == False]

print('Shape after removing:', listings_clean.shape)

In [ ]:
# investigate positive longitude flag
mask = listings_clean[listings_clean['pos_lon_flag'] == True]
print('# rows outside CA:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# set coordinate values to null
# even though cities correspond to their zipcodes, the lat/lon aren't within CA bounds
print('Before conversion:')
print(listings_clean.loc[
    listings_clean['pos_lon_flag'] == True,
    ['Latitude', 'Longitude']
].head())

# some coordinates seem to be misinputted, so we can flip longitude sign
listings_clean.loc[
    listings_clean['pos_lon_flag'] == True,
    'Longitude'
] *= -1

print('\nAfter conversion:')
print(listings_clean.loc[
    listings_clean['pos_lon_flag'] == True,
    ['Latitude', 'Longitude']
].head())

In [ ]:
# we can reflag the pos_lon_flag to see if the bool changes
print('Before reflagging:')
print(listings_clean['pos_lon_flag'].value_counts())

listings_clean['pos_lon_flag'] = listings_clean['Longitude'] > 0

print('\nAfter reflagging:')
print(listings_clean['pos_lon_flag'].value_counts())


In [ ]:
# also reflag oos coords
print('Before reflagging:')
print(listings_clean['oos_coords_flag'].value_counts())

listings_clean['oos_coords_flag'] = ~(listings_clean['Latitude'].between(32.0, 42.5) & listings_clean['Longitude'].between(-125.0, -113.5))

print('\nAfter reflagging:')
listings_clean['oos_coords_flag'].value_counts()

In [ ]:
# look into out of state coords
mask = listings_clean[listings_clean['oos_coords_flag'] == True]
print('# rows with out of state/implausible coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
listings_clean[(listings_clean['oos_coords_flag'] == True) &
               ~(listings_clean['Latitude'].notnull() &
               listings_clean['Longitude'].notnull())
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# remove non-cali coords via lat/lon
mask = (
        listings_clean['oos_coords_flag'] == True &
        ~(
            listings_clean['Latitude'].notnull() &
            listings_clean['Longitude'].notnull()
        )
        )

print('Shape before removing:', listings_clean.shape)
print('Shape of oos_cords_flag before removal:', listings_clean['oos_coords_flag'].shape)
listings_clean = listings_clean[mask]

print('Shape after removing:', listings_clean.shape)
print('Shape of oos_cords_flag after removal:', listings_clean['oos_coords_flag'].shape)

In [ ]:
listings_clean['oos_coords_flag'].value_counts()

In [ ]:
listings_clean[(listings_clean['oos_coords_flag'] == True) &
               (~listings_clean['PostalCode'].astype(str).str.startswith('9'))
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# remove non-cali via postal code
mask = (
    listings_clean['oos_coords_flag'] &
    ~listings_clean['PostalCode'].astype(str).str.startswith('9')
)

print("Rows to remove:", mask.sum())

print('Shape before removing:', listings_clean.shape)
print('Shape of oos_cords_flag before removal:', listings_clean['oos_coords_flag'].shape)
listings_clean = listings_clean[~mask]

print('Shape after removing:', listings_clean.shape)
print('Shape of oos_cords_flag after removal:', listings_clean['oos_coords_flag'].shape)

In [ ]:
listings_clean[listings_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']].isna().sum()

In [ ]:
listings_clean[listings_clean['oos_coords_flag']
               & listings_clean['City'].isna()
               ]['PostalCode'].unique()

In [ ]:
# convert nan cities according to zipcode
zip_to_city = {
    '92061': 'Pauma Valley',
    '94065': 'Redwood City',
    '95004': 'Aromas',
    '93933': 'Marina',
    '95670': 'Rancho Cordova',
    '92111': 'San Diego',
    '95703': 'Auburn',
    '91932': 'Imperial Beach',
    '94074': 'San Gregorio',
    '93908': 'Salinas',
    '91326': 'Porter Ranch',
    '91962': 'Pine Valley',
    '95419': 'Camp Meeker',
    '94904': 'Greenbrae',
    '92240': 'Desert Hot Springs',
    '91906': 'Campo',
    '94574': 'Saint Helena',
    '95497': 'The Sea Ranch',
    '95002': 'Alviso',
    '92128': 'San Diego',
    '94028': 'Portola Valley',
    '96022': 'Cottonwood',
    '90710': 'Harbor City',
    '92129': 'San Diego',
    '96137': 'Westwood',
    '92109': 'San Diego',
    '93291': 'Visalia',
    '95023': 'Hollister',
    '99999': np.nan,  # invalid zipcode
    '94516': 'Crockett',
    '95635': 'Greenwood',
    '95076': 'Watsonville',
    '95346': 'Mi Wuk Village',
    '92037': 'La Jolla',
    '92549': 'Idyllwild',
    '92676': 'Silverado',
    '93266': 'Stratford',
    '95628': 'Fair Oaks',
    '95664': 'Pilot Hill',
    '91411': 'Van Nuys',
    '95006': 'Boulder Creek',
    '92123': 'San Diego',
    '94062': 'Redwood City',
    '93222': 'Frazier Park',
    '94520': 'Concord',
    '92075': 'Solana Beach',
    '93015': 'Fillmore',
    '96136': 'Susanville',
    '95246': 'Linden',
    '95545': 'Honeydew',
    '95961': 'Olivehurst',
    '95614': 'Cool',
    '95364': 'Stevinson',
    '96048': 'Junction City',
    '93608': 'Cantua Creek',
    '94303': 'Palo Alto',
    '95385': 'Winton',
    '92024': 'Encinitas',
    '92028': 'Fallbrook',
    '96146': 'Olympic Valley',
    '95305': 'Coulterville',
    '95314': 'Cressey'
}

In [ ]:
# normalize zipcodes to remove the hyphens
listings_clean['PostalCode'] = (
    listings_clean['PostalCode']
    .astype(str)
    .str[:5]
)

# map zipcode to the correct city
listings_clean['City'] = (
    listings_clean['City']
    .fillna(
        listings_clean['PostalCode'].astype(str).map(zip_to_city)
    )
)

# confirm changes were made
listings_clean[listings_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']].isna().sum()

In [ ]:
listings_clean[listings_clean['oos_coords_flag']
               & listings_clean['City'].isna()
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# remove rows with 99999 postal code
print('Shape before removing:', listings_clean.shape)
print('Shape of oos_cords_flag before removal:', listings_clean['oos_coords_flag'].shape)

listings_clean = listings_clean[listings_clean['PostalCode'] != '99999']

print('Shape after removing:', listings_clean.shape)
print('Shape of oos_cords_flag after removal:', listings_clean['oos_coords_flag'].shape)

In [ ]:
listings_clean[listings_clean['oos_coords_flag'] == True
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# double check that changes were made
listings_clean[listings_clean['oos_coords_flag'] == True
               ][['City']].isna().sum()

### done reviewing, check flagged cols

In [ ]:
listings_clean.iloc[:,-8:].head(3)

In [ ]:
listings_clean['neg_livingarea_flag'].value_counts()
# neg_livingarea_flag
# False    606998
# Name: count, dtype: int64

In [ ]:
listings_clean[listings_clean['neg_dom_flag'] == True]['DaysOnMarket']

In [ ]:
listings_clean['negative_timeline_flag'].value_counts()
# negative_timeline_flag
# False    606694
# True        304
# Name: count, dtype: int64

listings_clean[listings_clean['negative_timeline_flag'] == True
               ][['ListingContractDate', 'PurchaseContractDate']].isna().sum()

In [ ]:
listings_clean['missing_coords_flag'].value_counts()
# missing_coords_flag
# False    526246
# True      80752
# Name: count, dtype: int64

listings_clean[listings_clean['missing_coords_flag'] == True][['Latitude', 'Longitude']].isna().sum()

In [ ]:
listings_clean['sentinel_coords_flag'].value_counts()
# sentinel_coords_flag
# False    606998
# Name: count, dtype: int64

listings_clean['pos_lon_flag'].value_counts()
# pos_lon_flag
# False    606998
# Name: count, dtype: int64

listings_clean['oos_coords_flag'].value_counts()
# oos_coords_flag
# False    526246
# True      80752
# Name: count, dtype: int64

In [ ]:
# drop flagged columns
print('Shape before removing:', listings_clean.shape)
listings_clean = listings_clean.drop(columns = ['neg_livingarea_flag',
                                        'neg_dom_flag',
                                        'negative_timeline_flag',
                                        'missing_coords_flag',
                                        'sentinel_coords_flag',
                                        'pos_lon_flag',
                                        'oos_coords_flag'])

print('Shape after removing:', listings_clean.shape)

### deliverable
- document every xformation made & why
- include:
    - before/after counts
    - dtype confirmations
    - date consistency flag counts
    - geographic data quality summary noting invalid coordinate records

### ``LISTINGS`` TRANSFORMATION SUMMARY
<hr>

> STARTING dataset shape: (607724, 86)

The following date fields were converted to datetime format:
- CloseDate
- PurchaseContractDate
- ListingContractDate
- ContractStatusChangeDate

In [ ]:
listings[date_cols].dtypes

The following __13__ columns were identified to contain __over 90% nulls__ and were removed:
- AboveGradeFinishedArea
- BelowGradeFinishedArea
- BuilderName
- BuildingAreaTotal
- BusinessType
- CoBuyerAgentFirstName
- CoveredSpaces
- ElementarySchoolDistrict
- FireplacesTotal
- LotSizeDimensions
- MiddleOrJuniorSchoolDistrict
- TaxAnnualAmount
- TaxYear

> Updated dataset shape: (607724, 73)

The following __15__ columns were considered and removed for having __over 50% nulls__:
- AssociationFeeFrequency
- BuyerAgencyCompensation
- BuyerAgencyCompensationType
- BuyerAgentFirstName
- BuyerAgentLastName
- BuyerAgentMlsId
- BuyerOfficeAOR
- BuyerOfficeName
- BuyerOfficeName.1
- CloseDate
- CloseDate.1
- CoListAgentFirstName
- CoListAgentLastName
- CoListOfficeName
- SubdivisionName

> Updated dataset shape: (607724, 58)

The remaining 58 columns were manually inspected and the following __10 were removed__ as they were considered to be unnecessary for analysis:
- PropertyType.1
- ListAgentFirstName.1
- DaysOnMarket.1
- LivingArea.1
- Longitude.1
- Latitude.1
- ListPrice.1
- ListAgentLastName.1
- UnparsedAddress.1
- ListAgentFirstName
- ListAgentLastName
- StreetNumberNumeric
- PropertyType
- ListAgentEmail
- LotSizeArea
- StateOrProvince
- ListingKeyNumeric

> Updated dataset shape: (607724, 41)

Rows were also cleaned by flagging for invalid values. Below is the count for each flag, how it was handled, and how it affected the overall dataset shape.

#### VALIDITY CHECKS
<hr>

- neg_livingarea_flag
    - count: 393
    - handled by: removal
    - notes: all 0-values
    > updated dataset shape: (607331, 48)
- neg_dom_flag
    - count: 32
    - handled by: converted to null
    - notes: attempts to fix negative value didn't work, maybe because PurchaseContractDate was inputted before ListingContractDate, so they were set to null so we can still use other information given by the row
    > updated dataset shape: (607331, 48) || UNCHANGED

There were no rows with negative close price, bedrooms, or bathrooms, so flag columns were not created for them

#### CONSISTENCY CHECKS
<hr>

- negative_timeline_flag
    - count: 304
    - handled by: converted to null
    > updated dataset shape: (607331, 48) || UNCHANGED

### GEOGRAPHIC CHECKS
<hr>

- missing_coords_flag
    - count: 80802
    - handled by: left them alone
    > updated dataset shape: (607331, 48) || UNCHANGED
- sentinel_coords_flag
    - count: 75
    - handled by: removal
    > updated dataset shape: (607256, 48)
- pos_lon_flag
    - count: 92
    - handled by: converted to null
    > updated dataset shape: (607256, 48) || UNCHANGED
- oos_coords_flag
    - count: 81007
    - handled by: removal
    - transformation 1: removed non-CA coords via lat/lon
    > updated dataset shape: (607051, 48)
    - transformation 2: removed non-CA coords via postal code
    > updated dataset shape: (607020, 48)
    - transformation 3: mapped null cities according to zipcode
    - transformation 4: normalized zipcodes to remove hyphens
    - transformation 5: removed rows with 9999 postal codes
    > updated dataset shape: (606998, 48)

In the end, all 7 flagged columns were dropped which leaves us with:
## final dataset shape: (606998, 41)

> recall starting dataset shape (607724, 86)

Other thoughts: While majority of the dataset has been cleaned, a small percentage of data were converted to nulls, especially when it comes to geographic data. When visualizing geographic data, these null values will be excluded and noted in future reports or presentations. The reason they are kept in the dataset is because those rows contain other useful information that can be visualized with no problem (e.g comparing list price to living area of a property)

In [ ]:
listings_clean.isna().sum()

In [ ]:
# save dataset as csv
listings_clean.to_csv('../data/listings_clean.csv', index = False)